In [0]:
!python --version

In [0]:
import requests
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.getOrCreate()


In [0]:
spark

In [0]:

API_KEY   = "SgAHaNKBbicyHtcX7HKRlWcV60HbXftJJ9IBQiD1"
BRONZE_DB = "petroleum_bronze"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")
print("Ready.")


In [0]:
# ──------------- Robust fetch with pagination ────────────────────────────────────
import math

def fetch_by_series_id(series_id: str, label: str):
    """
    Fetches ALL available rows for a series within the date range.
    Uses pagination so we never silently truncate data.
    """
    base_url = f"https://api.eia.gov/v2/seriesid/{series_id}"
    PAGE_SIZE = 500   # EIA max per request is 5000, 500 is safe and fast
    all_rows  = []
    offset    = 0

    while True:
        params = {
            "api_key":             API_KEY,
            "data[0]":             "value",
            "start":               "2024-01-01",
            "end":                 "2026-03-31",
            "sort[0][column]":     "period",
            "sort[0][direction]":  "asc",
            "length":              PAGE_SIZE,
            "offset":              offset,      # pagination offset
        }

        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        resp      = data.get("response", {})
        page_rows = resp.get("data", [])
        total     = int(resp.get("total", 0))  # EIA tells us total matching rows

        for record in page_rows:
            all_rows.append({
                "series_id":    series_id,
                "series_label": label,
                "date":         str(record.get("period", "")),
                "value":        str(record.get("value", "")),
                "unit":         str(record.get("units", "")),
                "ingested_at":  datetime.utcnow().isoformat()
            })

        offset += PAGE_SIZE

        # Stop when we've fetched everything
        if offset >= total or len(page_rows) == 0:
            break

    print(f"  {label}: {len(all_rows)} rows (total available: {total})")
    return all_rows

In [0]:
# ──— Series list using confirmed v1 series IDs ──────────────────────
# These are the original EIA v1 series IDs
SERIES = [
    { "series_id": "PET.RWTC.W",       "label": "WTI Crude Oil Spot Price ($/barrel)",           "table": "raw_wti_price" },
    { "series_id": "PET.RBRTE.W",      "label": "Brent Crude Oil Spot Price ($/barrel)",          "table": "raw_brent_price" },
    { "series_id": "PET.WCRFPUS2.W",   "label": "US Weekly Crude Oil Production (Mbbl/day)",      "table": "raw_us_production" },
    { "series_id": "PET.WCRSTUS1.W",   "label": "US Crude Oil Stocks (Million barrels)",           "table": "raw_crude_stocks" },
    { "series_id": "PET.WGFUPUS2.W",   "label": "US Weekly Product Supplied (Mbbl/day)",          "table": "raw_product_supplied" },
    { "series_id": "PET.EMM_EPMR_PTE_NUS_DPG.W",  "label": "US Gasoline Retail Price ($/gallon)", "table": "raw_gasoline_price" },
    { "series_id": "PET.EMD_EPD2D_PTE_NUS_DPG.W", "label": "US Diesel Retail Price ($/gallon)",   "table": "raw_diesel_price" },
]

In [0]:
# ──Ingest all series ───────────────────────────────────────────────
schema = StructType([
    StructField("series_id",    StringType(), True),
    StructField("series_label", StringType(), True),
    StructField("date",         StringType(), True),
    StructField("value",        StringType(), True),
    StructField("unit",         StringType(), True),
    StructField("ingested_at",  StringType(), True),
])

print("Starting Bronze ingestion...\n")
for s in SERIES:
    #rows = fetch_by_series_id(s["series_id"], s["label"], s["table"])
    rows = fetch_by_series_id(s["series_id"], s["label"])  # removed s["table"]
    
    if not rows:
        print(f"  WARNING: 0 rows for {s['series_id']} — skipping")
        continue

    df = spark.createDataFrame(rows, schema=schema)
    table_path = f"{BRONZE_DB}.{s['table']}"
    df.write.format("delta").mode("overwrite").saveAsTable(table_path)
    print(f"  Written: {table_path} ({df.count()} rows)\n")

print("Bronze ingestion complete!")

In [0]:
# ──Validation ──────────────────────────────────────────────────────
print("=== BRONZE VALIDATION ===")
for s in SERIES:
    try:
        df = spark.table(f"{BRONZE_DB}.{s['table']}")
        print(f"{s['table']:35s}  rows: {df.count()}")
    except:
        print(f"{s['table']:35s}  NOT FOUND")

# Preview one table
spark.table(f"{BRONZE_DB}.raw_wti_price").orderBy("date").show(5, truncate=False)

# preview all tables
query="select count(*) from petroleum_bronze.raw_brent_price limit 10"
spark.sql(query).show()